# Add Hebrew Line Names to Hubs Data

This notebook adds Hebrew line names to the hubs_with_complete_data output.

**Input files:**
1. Output from COMPLETE_TRANSIT_PIPELINE.ipynb (e.g., `hubs_with_complete_data.csv` or `grouped_hubs_ready_for_scoring_*.csv`)
2. CSV file with columns:
   - `LineName`: Line ID (matches values in `Line_Unique` column)
   - `Line_n_Mode`: Full Hebrew name for the line

**Process:**
1. Load the hubs data
2. Parse `Line_Unique` column (complex nested format)
3. Explode to create one row per line ID
4. Join with Hebrew names CSV
5. Group back by hub to create list of Hebrew names
6. Merge result into original dataframe

**Output:**
- Updated CSV with new `Line_Hebrew_Names` column

## Setup

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
from pathlib import Path
from IPython.display import display

# Setup paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RESULTS_DIR = DATA_DIR / "results"

# Create directories if needed
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## Step 1: Load Hubs Data

Load the output from COMPLETE_TRANSIT_PIPELINE which contains the `Line_Unique` column.

In [ ]:
# Configure input file path
# Option 1: Provide full path to your hubs file
hubs_file = NOTEBOOKS_DIR / 'grouped_hubs_ready_for_scoring_21082025.csv'

# Option 2: Or specify a different file
# hubs_file = RESULTS_DIR / 'hubs_with_complete_data.csv'

# Check if file exists
if not hubs_file.exists():
    print(f"ERROR: File not found: {hubs_file}")
    print("\nPlease update the 'hubs_file' variable above with the correct path.")
    raise FileNotFoundError(f"Hubs file not found: {hubs_file}")

print(f"Loading hubs data from: {hubs_file.name}")
hubs_df = pd.read_csv(hubs_file, encoding='utf-8')

print(f"\nLoaded {len(hubs_df)} hubs")
print(f"Columns: {len(hubs_df.columns)}")

# Check for required columns
if 'Line_Unique' not in hubs_df.columns:
    raise ValueError("'Line_Unique' column not found in hubs data")
if 'group' not in hubs_df.columns:
    raise ValueError("'group' column not found in hubs data")

print("\n✓ Required columns found: 'group' and 'Line_Unique'")

# Display sample
print("\nSample of Line_Unique column:")
display(hubs_df[['group', 'Line_Unique']].head(10))

## Step 2: Parse and Explode Line_Unique Column

The `Line_Unique` column has a complex nested format:
- Outer: Python list (as string)
- Inner: Numpy array-like strings with space-separated values

Example: `["['rail_1_1' 'rail_1_2']"]` should become `['rail_1_1', 'rail_1_2']`

We'll use explode-join-groupby pattern for correct mapping.

In [ ]:
def parse_line_unique_element(element_str):
    """
    Parse a single element from Line_Unique list.
    
    Input format: "['rail_1_1' 'rail_1_2']" (numpy array-like string)
    Output: ['rail_1_1', 'rail_1_2']
    
    Args:
        element_str: String element from outer list
    
    Returns:
        list: Individual line IDs
    """
    if pd.isna(element_str) or element_str == '':
        return []
    
    element_str = str(element_str).strip()
    
    # Remove outer brackets
    element_str = element_str.strip('[]')
    
    # Remove all quotes
    element_str = element_str.replace("'", "").replace('"', '')
    
    # Split by whitespace (handles space-separated values)
    lines = element_str.split()
    
    # Clean each line ID
    lines = [line.strip() for line in lines if line.strip()]
    
    return lines


def parse_line_unique_full(value):
    """
    Parse complete Line_Unique value.
    
    Input: ["['rail_1_1' 'rail_1_2']", "['rail_5_1' 'rail_5_2']"]
    Output: ['rail_1_1', 'rail_1_2', 'rail_5_1', 'rail_5_2']
    
    Args:
        value: Raw Line_Unique value
    
    Returns:
        list: Flat list of all unique line IDs
    """
    if pd.isna(value) or value == '' or value == '[]':
        return []
    
    # Parse outer list structure
    try:
        outer_list = ast.literal_eval(str(value))
        if not isinstance(outer_list, list):
            outer_list = [outer_list]
    except (ValueError, SyntaxError):
        # Fallback: treat as single string
        outer_list = [value]
    
    # Parse each element and flatten
    all_lines = []
    for element in outer_list:
        lines = parse_line_unique_element(element)
        all_lines.extend(lines)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_lines = []
    for line in all_lines:
        if line not in seen:
            seen.add(line)
            unique_lines.append(line)
    
    return unique_lines


# Test the parser with sample data
print("Testing parser with sample Line_Unique values:")
print("="*80)
for idx in range(min(10, len(hubs_df))):
    val = hubs_df.loc[idx, 'Line_Unique']
    parsed = parse_line_unique_full(val)
    print(f"\nRow {idx}:")
    print(f"  Original: {val}")
    print(f"  Parsed: {parsed}")
    print(f"  Count: {len(parsed)}")

## Step 3: Create Exploded DataFrame

Create one row per (group, line_id) pair for joining with Hebrew names.

In [ ]:
# Extract group and Line_Unique columns
print("Extracting group and Line_Unique columns...")
group_lines_df = hubs_df[['group', 'Line_Unique']].copy()

print(f"Starting with {len(group_lines_df)} hubs")

# Parse Line_Unique to list of line IDs
print("\nParsing Line_Unique column...")
group_lines_df['lines_list'] = group_lines_df['Line_Unique'].apply(parse_line_unique_full)

# Count total lines before explosion
total_lines_before = group_lines_df['lines_list'].apply(len).sum()
print(f"Total lines found: {total_lines_before}")

# Show distribution of line counts per hub
lines_per_hub = group_lines_df['lines_list'].apply(len)
print(f"\nLines per hub statistics:")
print(f"  Min: {lines_per_hub.min()}")
print(f"  Max: {lines_per_hub.max()}")
print(f"  Mean: {lines_per_hub.mean():.2f}")
print(f"  Median: {lines_per_hub.median():.0f}")

# Display sample
print("\nSample of parsed data:")
display(group_lines_df[['group', 'lines_list']].head(10))

In [ ]:
# Explode to create one row per line
print("Exploding to create one row per (group, line) pair...")

group_line_pairs = group_lines_df.explode('lines_list').copy()

# Rename column for clarity
group_line_pairs.rename(columns={'lines_list': 'LineName'}, inplace=True)

# Remove rows with empty line names
group_line_pairs = group_line_pairs[group_line_pairs['LineName'].notna()].copy()
group_line_pairs = group_line_pairs[group_line_pairs['LineName'] != ''].copy()

# Drop the original Line_Unique column
group_line_pairs = group_line_pairs[['group', 'LineName']].copy()

# Clean LineName
group_line_pairs['LineName'] = group_line_pairs['LineName'].str.strip()

print(f"\nCreated {len(group_line_pairs)} (group, line) pairs")
print(f"Unique groups: {group_line_pairs['group'].nunique()}")
print(f"Unique lines: {group_line_pairs['LineName'].nunique()}")

# Display sample
print("\nSample of exploded data:")
display(group_line_pairs.head(20))

# Show most common lines
print("\nTop 10 most common lines:")
top_lines = group_line_pairs['LineName'].value_counts().head(10)
display(pd.DataFrame({'LineName': top_lines.index, 'Count (hubs)': top_lines.values}))

## Step 4: Load Hebrew Line Names CSV

**Expected CSV format:**
- `LineName`: Line ID (e.g., 'rail_1_1', 'brt022')
- `Line_n_Mode`: Full Hebrew name for the line

**IMPORTANT:** Please provide the path to your Hebrew line names CSV file below.

In [ ]:
# ==========================================
# CONFIGURE THIS: Path to Hebrew line names CSV
# ==========================================

# Option 1: Place file in data directory
hebrew_names_file = DATA_DIR / 'hebrew_line_names.csv'

# Option 2: Or specify full path
# hebrew_names_file = Path('/path/to/your/hebrew_line_names.csv')

# ==========================================

# Check if file exists
if not hebrew_names_file.exists():
    print(f"ERROR: Hebrew line names file not found!")
    print(f"Expected location: {hebrew_names_file}")
    print(f"\nPlease provide a CSV file with these columns:")
    print(f"  - LineName: Line ID (e.g., 'rail_1_1', 'brt022')")
    print(f"  - Line_n_Mode: Full Hebrew name")
    print(f"\nUpdate the 'hebrew_names_file' variable in the cell above.")
    raise FileNotFoundError(f"Hebrew line names file not found: {hebrew_names_file}")

print(f"Loading Hebrew line names from: {hebrew_names_file.name}")
hebrew_names_df = pd.read_csv(hebrew_names_file, encoding='utf-8')

print(f"\nLoaded {len(hebrew_names_df)} line name records")
print(f"Columns: {hebrew_names_df.columns.tolist()}")

# Check for required columns
if 'LineName' not in hebrew_names_df.columns:
    raise ValueError("'LineName' column not found in Hebrew names file")
if 'Line_n_Mode' not in hebrew_names_df.columns:
    raise ValueError("'Line_n_Mode' column not found in Hebrew names file")

# Clean line names
hebrew_names_df['LineName'] = hebrew_names_df['LineName'].astype(str).str.strip()
hebrew_names_df['Line_n_Mode'] = hebrew_names_df['Line_n_Mode'].astype(str).str.strip()

# Remove duplicates (keep first)
hebrew_names_df = hebrew_names_df.drop_duplicates(subset='LineName', keep='first')

print("\n✓ Required columns found")
print(f"\nUnique line names: {hebrew_names_df['LineName'].nunique()}")

# Display sample
print("\nSample of Hebrew line names:")
display(hebrew_names_df[['LineName', 'Line_n_Mode']].head(20))

## Step 5: Join with Hebrew Names

Join the exploded (group, LineName) pairs with Hebrew names.

In [ ]:
# Join on LineName
print("Joining with Hebrew line names...")
print(f"Before join: {len(group_line_pairs)} records")

group_line_hebrew = group_line_pairs.merge(
    hebrew_names_df[['LineName', 'Line_n_Mode']],
    on='LineName',
    how='left'
)

print(f"After join: {len(group_line_hebrew)} records")

# Check for unmatched lines
unmatched = group_line_hebrew['Line_n_Mode'].isna().sum()
if unmatched > 0:
    print(f"\n⚠️  WARNING: {unmatched} line-group pairs have no Hebrew name")
    print(f"   This represents {unmatched / len(group_line_hebrew) * 100:.1f}% of all pairs")
    
    # Show unmatched lines
    unmatched_lines = group_line_hebrew[group_line_hebrew['Line_n_Mode'].isna()]['LineName'].unique()
    print(f"\n   Unmatched line IDs ({len(unmatched_lines)} unique):")
    for line in sorted(unmatched_lines)[:20]:
        print(f"     - {line}")
    if len(unmatched_lines) > 20:
        print(f"     ... and {len(unmatched_lines) - 20} more")
else:
    print("\n✅ All lines matched successfully!")

# Display sample
print("\nSample of joined data:")
display(group_line_hebrew.head(20))

## Step 6: Group by Hub to Create Hebrew Names List

Aggregate back to one row per hub with list of all Hebrew names.

In [ ]:
# For unmatched lines, use original LineName as fallback
group_line_hebrew['Line_n_Mode_Final'] = group_line_hebrew['Line_n_Mode'].fillna(
    '[Unknown] ' + group_line_hebrew['LineName']
)

# Group by hub and aggregate Hebrew names into list
print("Grouping by hub to create Hebrew names list...")

hebrew_names_by_hub = group_line_hebrew.groupby('group').agg({
    'Line_n_Mode_Final': lambda x: list(x)
}).reset_index()

hebrew_names_by_hub.rename(columns={'Line_n_Mode_Final': 'Line_Hebrew_Names'}, inplace=True)

print(f"\nCreated Hebrew names list for {len(hebrew_names_by_hub)} hubs")

# Show statistics
names_per_hub = hebrew_names_by_hub['Line_Hebrew_Names'].apply(len)
print(f"\nHebrew names per hub:")
print(f"  Min: {names_per_hub.min()}")
print(f"  Max: {names_per_hub.max()}")
print(f"  Mean: {names_per_hub.mean():.2f}")
print(f"  Median: {names_per_hub.median():.0f}")

# Display sample
print("\nSample of grouped data:")
display(hebrew_names_by_hub.head(10))

## Step 7: Merge Back into Original DataFrame

Add the `Line_Hebrew_Names` column to the original hubs dataframe.

In [ ]:
# Merge with original dataframe
print("Merging Hebrew names into original dataframe...")
print(f"Hubs before merge: {len(hubs_df)}")

hubs_with_hebrew = hubs_df.merge(
    hebrew_names_by_hub,
    on='group',
    how='left'
)

print(f"Hubs after merge: {len(hubs_with_hebrew)}")

# Fill NaN with empty list for hubs without lines
hubs_with_hebrew['Line_Hebrew_Names'] = hubs_with_hebrew['Line_Hebrew_Names'].apply(
    lambda x: x if isinstance(x, list) else []
)

print(f"\n✓ Hebrew names column added")

# Count hubs with Hebrew names
hubs_with_names = (hubs_with_hebrew['Line_Hebrew_Names'].apply(len) > 0).sum()
print(f"\nHubs with at least one Hebrew name: {hubs_with_names} / {len(hubs_with_hebrew)}")

# Display sample
print("\nSample with Hebrew names:")
display_cols = ['group', 'Line_Hebrew_Names']
if 'address' in hubs_with_hebrew.columns:
    display_cols.insert(1, 'address')
display(hubs_with_hebrew[display_cols].head(10))

## Step 8: Show Detailed Examples

Display detailed line ID → Hebrew name mapping for verification.

In [ ]:
# Show detailed examples with original Line_Unique for comparison
print("Detailed mapping examples:")
print("="*80)

for idx in range(min(5, len(hubs_with_hebrew))):
    group = hubs_with_hebrew.loc[idx, 'group']
    line_unique = hubs_with_hebrew.loc[idx, 'Line_Unique']
    hebrew_names = hubs_with_hebrew.loc[idx, 'Line_Hebrew_Names']
    
    # Parse original to compare
    parsed_ids = parse_line_unique_full(line_unique)
    
    print(f"\nGroup {group}:")
    print(f"  Original Line_Unique: {line_unique}")
    print(f"  Parsed to {len(parsed_ids)} line IDs: {parsed_ids}")
    print(f"  Hebrew names ({len(hebrew_names)}):")
    for i, name in enumerate(hebrew_names):
        line_id = parsed_ids[i] if i < len(parsed_ids) else 'N/A'
        print(f"    {line_id} → {name}")

# Overall statistics
total_hebrew_names = hubs_with_hebrew['Line_Hebrew_Names'].apply(len).sum()
avg_per_hub = total_hebrew_names / len(hubs_with_hebrew)

print(f"\n" + "="*80)
print(f"Summary Statistics:")
print(f"  Total hubs: {len(hubs_with_hebrew)}")
print(f"  Total line-hub mappings: {total_hebrew_names}")
print(f"  Average Hebrew names per hub: {avg_per_hub:.2f}")
print(f"  Hubs with at least one name: {hubs_with_names}")

## Step 9: Export Results

Save the updated dataframe with the new `Line_Hebrew_Names` column.

In [ ]:
# Create output filename
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
output_file = RESULTS_DIR / f'hubs_with_hebrew_line_names_{timestamp}.csv'

# Prepare export dataframe
export_df = hubs_with_hebrew.copy()

# Convert Line_Hebrew_Names list to string for CSV export
export_df['Line_Hebrew_Names'] = export_df['Line_Hebrew_Names'].apply(str)

# Save to CSV
export_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ Results exported to: {output_file.name}")
print(f"\nFull path: {output_file}")
print(f"\nExport summary:")
print(f"  Total hubs: {len(export_df)}")
print(f"  Total columns: {len(export_df.columns)}")
print(f"\nNew column added:")
print(f"  - Line_Hebrew_Names: List of Hebrew names for all lines in each hub")

## Summary

✓ Processing complete!

### What was done:

1. **Loaded hubs data** with `Line_Unique` column
2. **Parsed Line_Unique** from complex nested format to flat list of line IDs
   - Format: `["['rail_1_1' 'rail_1_2']"]` → `['rail_1_1', 'rail_1_2']`
3. **Exploded to (group, LineName) pairs** - one row per line per hub
4. **Joined with Hebrew names CSV** (LineName → Line_n_Mode)
5. **Grouped back by hub** to create list of Hebrew names
6. **Merged into original dataframe** - added `Line_Hebrew_Names` column
7. **Exported results** to CSV file

### Output file:

- Location: `data/results/hubs_with_hebrew_line_names_YYYYMMDD_HHMM.csv`
- Contains all original columns plus:
  - `Line_Hebrew_Names`: List of Hebrew names for all lines (as string)

### Next steps:

- Review the output file to verify Hebrew names are correct
- If there were unmatched lines (marked with "[Unknown]"), add them to the Hebrew names CSV and re-run
- Use the `Line_Hebrew_Names` column for visualization or reporting

### Note on parsing:

The parser correctly handles:
- Single elements: `["['rail_1_1' 'rail_1_2']"]` → `['rail_1_1', 'rail_1_2']`
- Multiple elements: `["['brt021' 'brt022']", "['brt021']"]` → `['brt021', 'brt022']` (deduplicated)
- Space-separated numpy array format
- Automatic deduplication of line IDs